# Massa-mola: Definicao de derivada, Euler e RK4

Neste notebook, o foco e didatico em **posicao x(t)**.
A ultima celula mostra no mesmo bloco:
- aparato massa-mola oscilando;
- as 3 curvas de posicao sendo formadas.
    


## Aparato experimental usado

- parede fixa;
- mola ideal ligada a uma massa que oscila em 1D;
- sem amortecimento e sem forca externa.

Equacao modelada:
`x'' + (k/m) * x = 0`
    


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

plt.rcParams['animation.html'] = 'jshtml'
    


In [ ]:
# BLOCO 1 - Parametros
m = 1.0
k = 1.0
x0 = 1.0
v0 = 0.0
dt = 0.01
t_final = 20.0
t = np.arange(0.0, t_final + dt, dt)


def mostrar_amostra(nome, t, x, n=10):
    print(f"\nSaida numerica - {nome}")
    print(' i   t(s)      x(m)')
    for i in range(min(n, len(t))):
        print(f"{i:2d}  {t[i]:6.3f}   {x[i]:10.6f}")
    


In [ ]:
# BLOCO 2 - Definicao de derivada (diferenca finita de 2a ordem)
def simular_definicao_derivada(t, m, k, x0, v0):
    dt = t[1] - t[0]
    n = len(t)
    x = np.zeros(n)

    x[0] = x0
    a0 = -(k / m) * x0
    x[1] = x0 + v0 * dt + 0.5 * a0 * dt**2

    for i in range(1, n - 1):
        a_i = -(k / m) * x[i]
        x[i + 1] = 2 * x[i] - x[i - 1] + a_i * dt**2

    return x


# BLOCO 3 - Euler
def simular_euler(t, m, k, x0, v0):
    dt = t[1] - t[0]
    n = len(t)
    x = np.zeros(n)
    v = np.zeros(n)

    x[0] = x0
    v[0] = v0

    for i in range(n - 1):
        a = -(k / m) * x[i]
        v[i + 1] = v[i] + a * dt
        x[i + 1] = x[i] + v[i] * dt

    return x


# BLOCO 4 - RK4
def derivadas(x, v, m, k):
    return v, -(k / m) * x


def simular_rk4(t, m, k, x0, v0):
    dt = t[1] - t[0]
    n = len(t)
    x = np.zeros(n)
    v = np.zeros(n)

    x[0] = x0
    v[0] = v0

    for i in range(n - 1):
        xi, vi = x[i], v[i]

        k1_x, k1_v = derivadas(xi, vi, m, k)
        k2_x, k2_v = derivadas(xi + 0.5 * dt * k1_x, vi + 0.5 * dt * k1_v, m, k)
        k3_x, k3_v = derivadas(xi + 0.5 * dt * k2_x, vi + 0.5 * dt * k2_v, m, k)
        k4_x, k4_v = derivadas(xi + dt * k3_x, vi + dt * k3_v, m, k)

        x[i + 1] = xi + (dt / 6.0) * (k1_x + 2 * k2_x + 2 * k3_x + k4_x)
        v[i + 1] = vi + (dt / 6.0) * (k1_v + 2 * k2_v + 2 * k3_v + k4_v)

    return x
    


In [ ]:
# BLOCO 5 - Simulacoes
x_def = simular_definicao_derivada(t, m, k, x0, v0)
x_euler = simular_euler(t, m, k, x0, v0)
x_rk4 = simular_rk4(t, m, k, x0, v0)

mostrar_amostra('Definicao de derivada', t, x_def)
mostrar_amostra('Euler', t, x_euler)
mostrar_amostra('RK4', t, x_rk4)
    


## Mesma celula: aparato oscilando + 3 curvas de posicao se formando

No painel esquerdo, o aparato massa-mola e animado usando RK4.
No painel direito, as tres curvas (`Def. derivada`, `Euler`, `RK4`) aparecem em tempo real.
    


In [ ]:
def desenhar_mola(x_massa, x_parede=-1.2, espiras=12, amplitude=0.08, pontos=220):
    x_final = x_massa - 0.08
    if x_final <= x_parede + 0.05:
        x_final = x_parede + 0.05

    xs = np.linspace(x_parede, x_final, pontos)
    fase = np.linspace(0.0, 2.0 * np.pi * espiras, pontos)
    ys = amplitude * np.sin(fase)
    ys[0] = 0.0
    ys[-1] = 0.0
    return xs, ys


def animar_aparato_e_curvas(t, x_aparato, x_def, x_euler, x_rk4, titulo='Massa-mola'):
    max_frames = 420
    passo = max(1, len(t) // max_frames)

    t_anim = t[::passo]
    xa = x_aparato[::passo]
    xd = x_def[::passo]
    xe = x_euler[::passo]
    xr = x_rk4[::passo]

    fig, (ax_s, ax_g) = plt.subplots(1, 2, figsize=(12, 4.8))
    fig.suptitle(titulo)

    # Painel esquerdo: aparato
    ax_s.set_title('Aparato experimental: massa-mola')
    ax_s.set_xlabel('x (m)')
    ax_s.set_ylabel('y')
    xmin = min(-1.3, np.min(xa) - 0.3)
    xmax = max(1.3, np.max(xa) + 0.3)
    ax_s.set_xlim(xmin, xmax)
    ax_s.set_ylim(-0.4, 0.4)
    ax_s.grid(alpha=0.3)
    ax_s.axvline(-1.2, color='gray', lw=3)

    linha_mola, = ax_s.plot([], [], color='tab:blue', lw=2)
    massa, = ax_s.plot([], [], 'o', color='tab:red', ms=14)

    # Painel direito: 3 curvas se formando
    amp = max(np.max(np.abs(np.concatenate([xd, xe, xr]))) * 1.25, 0.2)
    ax_g.set_xlim(0, t[-1])
    ax_g.set_ylim(-amp, amp)
    ax_g.set_title('x(t): Def. derivada, Euler e RK4')
    ax_g.set_xlabel('tempo (s)')
    ax_g.set_ylabel('x (m)')
    ax_g.grid(alpha=0.3)

    linha_def, = ax_g.plot([], [], color='black', linestyle='--', linewidth=2.0, label='Def. derivada', zorder=4)
    linha_eul, = ax_g.plot([], [], color='tab:orange', linewidth=1.8, label='Euler', zorder=3)
    linha_rk, = ax_g.plot([], [], color='tab:green', linewidth=1.8, label='RK4', zorder=2)

    pt_def, = ax_g.plot([], [], 'o', color='black', ms=4)
    pt_eul, = ax_g.plot([], [], 'o', color='tab:orange', ms=4)
    pt_rk, = ax_g.plot([], [], 'o', color='tab:green', ms=4)
    ax_g.legend(loc='upper right')

    def init():
        linha_mola.set_data([], [])
        massa.set_data([], [])
        linha_def.set_data([], [])
        linha_eul.set_data([], [])
        linha_rk.set_data([], [])
        pt_def.set_data([], [])
        pt_eul.set_data([], [])
        pt_rk.set_data([], [])
        return linha_mola, massa, linha_def, linha_eul, linha_rk, pt_def, pt_eul, pt_rk

    def update(i):
        xs, ys = desenhar_mola(xa[i])
        linha_mola.set_data(xs, ys)
        massa.set_data([xa[i]], [0.0])

        tt = t_anim[: i + 1]
        yd = xd[: i + 1]
        ye = xe[: i + 1]
        yr = xr[: i + 1]

        linha_def.set_data(tt, yd)
        linha_eul.set_data(tt, ye)
        linha_rk.set_data(tt, yr)

        pt_def.set_data([tt[-1]], [yd[-1]])
        pt_eul.set_data([tt[-1]], [ye[-1]])
        pt_rk.set_data([tt[-1]], [yr[-1]])
        return linha_mola, massa, linha_def, linha_eul, linha_rk, pt_def, pt_eul, pt_rk

    ani = FuncAnimation(fig, update, frames=len(t_anim), init_func=init, interval=30, blit=False)
    plt.close(fig)
    return HTML(ani.to_jshtml())


display(animar_aparato_e_curvas(t, x_rk4, x_def, x_euler, x_rk4, titulo='Massa-mola: aparato + curvas'))
    
